# Training YOLO11 — Dataset TJ v2_clahe = model (d) YOLO + CLAHE/gamma (Tahap 4)
Jalankan sel dari atas ke bawah (Run All).
Syarat: **Accelerator = GPU (T4 / P100)**, **Internet = On**, dataset `v2_clahe.zip` sudah di-**Add Input**.

Isi dataset v2: train 1.072, valid 168, test 161 (QC penuh, 319 gambar berkotak dobel dikecualikan), split per kamera-tanggal, 7 kelas.
Keluaran (di `/kaggle/working`, download `hasil_training.zip`):
- `runs/tj_v2_clahe/weights/best.pt` → model untuk inference di laptop
- `metrik_per_kelas.csv` → mAP50, mAP50-95, precision, recall per kelas, untuk valid/test dan test siang/malam
- grafik `results.png`, `confusion_matrix.png`, dll.

In [ ]:
# 1. Cari dataset & salin ke /kaggle/working (folder input read-only, ultralytics perlu menulis cache label)
import glob, shutil
from pathlib import Path
semua = glob.glob('/kaggle/input/**/data.yaml', recursive=True)
cari = [f for f in semua if open(f).readline().startswith('# Dataset TJ v2_clahe:')]
print('data.yaml ditemukan:', semua)
assert len(cari) == 1, f'Harus ada tepat SATU dataset v2_clahe di Input, ketemu {len(cari)}. Cek panel Input (hapus v1 lama / upload v2_clahe.zip).'
SRC = Path(cari[0]).parent
DS = Path('/kaggle/working/v2_clahe')
if DS.exists():
    shutil.rmtree(DS)  # selalu salin ulang, biar tidak tersisa data lama
shutil.copytree(SRC, DS)
for s in ['train', 'valid', 'test']:
    print(s, len(list((DS / 'images' / s).glob('*.jpg'))), 'gambar')
print(open(DS / 'data.yaml').read())

In [ ]:
# 2. Pasang ultralytics
!pip -q install ultralytics
import ultralytics; ultralytics.checks()

In [ ]:
# 3. Pengaturan (ubah di sini saja)
MODEL    = 'yolo11s.pt'   # 's' = lebih akurat untuk objek kecil; ganti 'yolo11n.pt' kalau inference CPU laptop terlalu lambat
EPOCHS   = 100
PATIENCE = 30             # berhenti otomatis kalau mAP valid tidak naik 30 epoch
IMGSZ    = 640            # gambar asli 640x360
BATCH    = 16
SEED     = 0
NAMA     = ['tj_bus', 'bus_lain', 'car', 'motorcycle', 'truck', 'mikrotrans', 'kend_prioritas']
UTAMA    = [0, 1, 2, 3, 4]  # kelas yang dilaporkan di metrik utama; mikrotrans & kend_prioritas terlalu sedikit (<=4 di valid/test)

In [ ]:
# 4. Training (±30-60 menit di T4 untuk 100 epoch)
from ultralytics import YOLO
model = YOLO(MODEL)
model.train(data=str(DS / 'data.yaml'), epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
            seed=SEED, deterministic=True, workers=2, cache='ram',
            project='/kaggle/working/runs', name='tj_v2_clahe', exist_ok=True, plots=True)

In [ ]:
# 5. Evaluasi best.pt: valid, test, test siang, test malam
import pandas as pd
BEST = '/kaggle/working/runs/tj_v2_clahe/weights/best.pt'
best = YOLO(BEST)

def yaml_eval(val_list):
    # yaml sementara: val = daftar gambar tertentu (txt di folder dataset)
    p = Path(f'/kaggle/working/eval_{Path(val_list).stem}.yaml')
    NL = chr(10)
    names = ''.join(f'  {i}: {n}' + NL for i, n in enumerate(NAMA))
    p.write_text(NL.join([f'path: {DS}', 'train: images/train', f'val: {val_list}', 'names:', names]))
    return str(p)

uji = {
    'valid':       (str(DS / 'data.yaml'), 'val'),
    'test':        (str(DS / 'data.yaml'), 'test'),
    'test_siang':  (yaml_eval('test_siang.txt'), 'val'),
    'test_malam':  (yaml_eval('test_malam.txt'), 'val'),
    'valid_siang': (yaml_eval('valid_siang.txt'), 'val'),
    'valid_malam': (yaml_eval('valid_malam.txt'), 'val'),
}
rows = []
for nama_uji, (y, split) in uji.items():
    r = best.val(data=y, split=split, imgsz=IMGSZ, batch=BATCH, plots=(nama_uji in ('valid', 'test')),
                 project='/kaggle/working/runs', name=f'eval_{nama_uji}', exist_ok=True, verbose=False)
    nt = getattr(r, 'nt_per_class', None)
    for j, c in enumerate(r.box.ap_class_index):
        p, rc, ap50, ap = r.box.class_result(j)
        rows.append(dict(uji=nama_uji, kelas=NAMA[c], kotak=int(nt[c]) if nt is not None else None,
                         precision=p, recall=rc, mAP50=ap50, mAP50_95=ap))
df = pd.DataFrame(rows)
# baris ringkasan: rata-rata kelas utama (0-4)
ring = (df[df.kelas.isin([NAMA[i] for i in UTAMA])].groupby('uji', sort=False)[['precision', 'recall', 'mAP50', 'mAP50_95']]
        .mean().reset_index().assign(kelas='RATA2_5_kelas_utama'))
df = pd.concat([df, ring], ignore_index=True)
df.to_csv('/kaggle/working/metrik_per_kelas.csv', index=False)
pd.set_option('display.float_format', '{:.3f}'.format)
display(ring)
display(df[df.uji == 'test'])

In [ ]:
# 6. Grafik training & confusion matrix test
from IPython.display import Image, display
for f in ['runs/tj_v2_clahe/results.png', 'runs/eval_test/confusion_matrix_normalized.png', 'runs/eval_test/PR_curve.png']:
    p = Path('/kaggle/working') / f
    if p.exists():
        print(f); display(Image(filename=str(p), width=900))

In [ ]:
# 7. Contoh prediksi di 6 gambar test (3 siang, 3 malam)
import random
random.seed(1)
pilih = []
for w in ['siang', 'malam']:
    lst = [l.strip().replace('./', str(DS) + '/', 1) for l in open(DS / f'test_{w}.txt')]
    pilih += random.sample(lst, 3)
for r in best.predict(pilih, imgsz=IMGSZ, conf=0.25, verbose=False):
    display(Image(data=__import__('cv2').imencode('.jpg', r.plot())[1].tobytes(), width=800))

In [ ]:
# 8. Zip hasil untuk di-download (Output -> hasil_training.zip)
!cd /kaggle/working && rm -f hasil_training.zip && zip -rq hasil_training.zip runs metrik_per_kelas.csv -x "runs/*/weights/last.pt"
!ls -lh /kaggle/working/hasil_training.zip